# Pixel3DMM Safe Rerun — Hair App Milestone 1

과거 Colab 오류와 2026-06-23 기준 공식 Pixel3DMM 코드를 다시 대조해 만든 안전 재실행 노트북이다.

- audited upstream commit: `fcd1fa973c7715b02a8948dfc679dff53cf85924`
- verified previously: A100 환경 빌드, crop, segmentation
- fixed here: A100/H100 arch 자동 감지, Torch pin, MICA 완전 우회, FLAME2020+2023 수동 설치/검증, output count 검증, multi-image tracking 인자 수정, raw log 저장
- license: Pixel3DMM CC BY-NC 4.0, FLAME 별도 연구 라이선스

> 위에서 아래로 실행한다. `condacolab.install()` 뒤 런타임이 재시작되는 것은 정상이다.
> private 사진, mesh, notebook output은 git에 올리지 않는다.
> FLAME2020.zip과 FLAME2023.zip은 본인이 FLAME 사이트에서 약관 동의 후 받아야 한다.

## 0. GPU 확인 및 CUDA arch 자동 기록

In [ ]:
!nvidia-smi
import pathlib, torch
assert torch.cuda.is_available(), 'GPU runtime이 아님: 런타임 유형을 GPU로 변경하세요.'
GPU_NAME = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
TORCH_ARCH = f'{cc[0]}.{cc[1]}+PTX'
pathlib.Path('/content/p3dmm_torch_arch.txt').write_text(TORCH_ARCH)
print('GPU:', GPU_NAME, '| compute capability:', cc, '| TORCH_CUDA_ARCH_LIST:', TORCH_ARCH)
assert cc in {(8, 0), (9, 0)}, '이 노트북은 A100 또는 H100 기준. 다른 GPU면 arch/dependency를 재검증하세요.'

## 1. Conda 설치 — 이 셀 뒤 런타임 자동 재시작

In [ ]:
!pip -q install condacolab
import condacolab
condacolab.install()

## 2. 공식 저장소 clone 및 audited commit 고정

In [ ]:
import condacolab; condacolab.check()
import os, pathlib
%cd /content
if not os.path.exists('/content/pixel3dmm/.git'):
    !git clone https://github.com/SimonGiebenhain/pixel3dmm.git
%cd /content/pixel3dmm
!git fetch origin
!git checkout fcd1fa973c7715b02a8948dfc679dff53cf85924
!git rev-parse HEAD

## 3. p3dmm 환경 생성 및 CUDA extension 빌드

공식 environment의 Torch 2.7/cu118 조합을 명시적으로 고정한다. 빌드는 10~25분 걸릴 수 있다.

In [ ]:
%%bash
set -euo pipefail
if ! conda env list | awk '{print $1}' | grep -qx p3dmm; then
  conda create -n p3dmm python=3.9 -y
fi
conda run -n p3dmm pip install \
  torch==2.7.0+cu118 torchvision==0.22.0+cu118 torchaudio==2.7.0+cu118 \
  --index-url https://download.pytorch.org/whl/cu118
conda install -n p3dmm -y \
  nvidia/label/cuda-11.8.0::cuda-nvcc nvidia/label/cuda-11.8.0::cuda-cccl \
  nvidia/label/cuda-11.8.0::cuda-cudart nvidia/label/cuda-11.8.0::cuda-cudart-dev \
  nvidia/label/cuda-11.8.0::libcusparse nvidia/label/cuda-11.8.0::libcusparse-dev \
  nvidia/label/cuda-11.8.0::libcublas nvidia/label/cuda-11.8.0::libcublas-dev \
  nvidia/label/cuda-11.8.0::libcurand nvidia/label/cuda-11.8.0::libcurand-dev \
  nvidia/label/cuda-11.8.0::libcusolver nvidia/label/cuda-11.8.0::libcusolver-dev
conda run -n p3dmm nvcc --version

In [ ]:
%%bash
set -euo pipefail
ENVDIR=/usr/local/envs/p3dmm
ARCH=$(cat /content/p3dmm_torch_arch.txt)
export CUDA_HOME=$ENVDIR
export TORCH_CUDA_ARCH_LIST=$ARCH
conda run -n p3dmm pip install ninja fvcore iopath
CUDA_HOME=$ENVDIR TORCH_CUDA_ARCH_LIST=$ARCH \
  conda run -n p3dmm pip install --no-build-isolation 'git+https://github.com/facebookresearch/pytorch3d.git@75ebeeaea0908c5527e7b1e305fbc7681382db47'
CUDA_HOME=$ENVDIR TORCH_CUDA_ARCH_LIST=$ARCH \
  conda run -n p3dmm pip install --no-build-isolation 'git+https://github.com/NVlabs/nvdiffrast.git@253ac4fcea7de5f396371124af597e6cc957bfae'
cd /content/pixel3dmm
conda run -n p3dmm pip install -r requirements.txt
conda run -n p3dmm pip install -e .
conda run -n p3dmm python - <<'PY'
import torch, pytorch3d, nvdiffrast.torch as dr, pixel3dmm
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'gpu', torch.cuda.get_device_name(0))
print('core imports: PASS')
PY

## 4. 전처리 의존성 설치 — 과거 오류 수정 포함

- SSH clone 대신 HTTPS
- FaceBoxes 전에 Cython 설치
- checkpoint를 공식 코드가 읽는 `pretrained_weights/`에 저장
- `ignore_mica=True`인데도 원본 tracker가 MICA 파일을 강제로 읽는 부분을 zero prior로 패치
- 전처리에서 불필요한 MICA 실행을 건너뜀

In [ ]:
%%bash
set -euo pipefail
PRE=/content/pixel3dmm/src/pixel3dmm/preprocessing
conda run -n p3dmm pip install -q Cython gdown
rm -rf "$PRE/facer" "$PRE/PIPNet"
cd "$PRE"
git clone https://github.com/FacePerceiver/facer.git
cd facer
git checkout ddd35c76ff840174b8a5403ad1c1255e37b8782b
cp ../replacement_code/farl.py facer/face_parsing/farl.py
cp ../replacement_code/facer_transform.py facer/transform.py
conda run -n p3dmm pip install -e .
cd "$PRE"
git clone https://github.com/jhb86253817/PIPNet.git
cd PIPNet
git checkout b9eab58816437403a34aa5bc3adeafe5081fd36b
cd ..
cd PIPNet/FaceBoxesV2/utils
conda run -n p3dmm sh make.sh
cd "$PRE/PIPNet"
mkdir -p snapshots/WFLW/pip_32_16_60_r18_l2_l1_10_1_nb10
conda run -n p3dmm gdown 1nVkaSbxy3NeqblwMTGvLg4nF49cI_99C \
  -O snapshots/WFLW/pip_32_16_60_r18_l2_l1_10_1_nb10/epoch59.pth
mkdir -p /content/pixel3dmm/pretrained_weights "$PRE/MICA/data"
cd /content/pixel3dmm/pretrained_weights
conda run -n p3dmm gdown 1SDV_8_qWTe__rX_8e4Fi-BE3aES0YzJY -O uv.ckpt
conda run -n p3dmm gdown 1KYYlpN-KGrYMVcAOT22NkVQC0UAfycMD -O normals.ckpt
test $(stat -c%s uv.ckpt) -gt 1000000
test $(stat -c%s normals.ckpt) -gt 1000000
test $(stat -c%s "$PRE/PIPNet/snapshots/WFLW/pip_32_16_60_r18_l2_l1_10_1_nb10/epoch59.pth") -gt 1000000
echo 'preprocessing dependencies/checkpoints: PASS'

In [ ]:
from pathlib import Path
import re

# 1) facer 최신 torch index dtype 오류
farl = Path('/content/pixel3dmm/src/pixel3dmm/preprocessing/facer/facer/face_parsing/farl.py')
text = farl.read_text()
text = text.replace("images[data['image_ids']]", "images[data['image_ids'].long()]")
farl.write_text(text)
assert "images[data['image_ids'].long()]" in farl.read_text()

# 2) ignore_mica=True 경로에서는 전처리 MICA 실행 자체를 생략
pre = Path('/content/pixel3dmm/scripts/run_preprocessing.py')
text = pre.read_text()
mica_call = "    os.system(f'cd {env_paths.CODE_BASE}/src/pixel3dmm/preprocessing/MICA ; python demo.py -video_name {vid_name} -a {env_paths.PREPROCESSED_DATA}/{vid_name}/arcface/')"
assert mica_call in text or 'HAIR_APP_SKIP_MICA' in text
text = text.replace(mica_call, "    print('HAIR_APP_SKIP_MICA: ignore_mica=True tracking will use a zero shape prior')")
pre.write_text(text)

# 3) upstream tracker는 ignore_mica=True여도 mica/identity.npy를 먼저 읽으므로 zero prior로 우회
tracker = Path('/content/pixel3dmm/src/pixel3dmm/tracking/tracker.py')
text = tracker.read_text()
pattern = re.compile(r"        mica_folder = f'\{DATA_FOLDER\}/mica'.*?            mica_shape = np.mean\(mica_shapes, axis=0\)\n", re.S)
replacement = """        # HAIR_APP_ZERO_MICA_PRIOR
        if self.config.ignore_mica:
            mica_shape = np.zeros(self.config.num_shape_params, dtype=np.float32)
        else:
            mica_folder = f'{DATA_FOLDER}/mica'
            mica_files = os.listdir(mica_folder)
            mica_shapes = []
            for mica_file in mica_files:
                one_shape = np.load(f'{mica_folder}/{mica_file}/identity.npy')
                mica_shapes.append(np.squeeze(one_shape))
            mica_shapes = np.stack(mica_shapes, axis=0)
            mica_shape = mica_shapes[0, :] if self.config.early_exit else np.mean(mica_shapes, axis=0)
"""
if 'HAIR_APP_ZERO_MICA_PRIOR' not in text:
    text, changed = pattern.subn(replacement, text)
    assert changed == 1, f'tracker patch count={changed}'
tracker.write_text(text)
print('facer + skip-MICA + tracker zero-prior patches: PASS')

## 5. Google Drive 마운트 및 FLAME2020 + FLAME2023 수동 설치

1. <https://flame.is.tue.mpg.de/> 로그인 및 약관 동의
2. `FLAME2020.zip`, `FLAME2023.zip`을 직접 다운로드
3. Drive `MyDrive/hair_app/models/`에 zip 그대로 업로드

자동 wget은 로그인 HTML을 zip처럼 저장할 수 있어 사용하지 않는다. Pixel3DMM은 FLAME2023 mesh를 쓸 때도 FLAME2020의 generic model, landmark, masks를 읽으므로 **두 zip 모두 필수**다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, zipfile
models = Path('/content/drive/MyDrive/hair_app/models')
models.mkdir(parents=True, exist_ok=True)
assets = Path('/content/pixel3dmm/src/pixel3dmm/preprocessing/MICA/data')

def install_flame(zip_name, marker, destination, required):
    z = models / zip_name
    assert z.exists(), f'{z} 없음: FLAME 사이트에서 직접 받아 Drive에 업로드하세요.'
    assert z.stat().st_size > 100_000, f'{z}가 너무 작음({z.stat().st_size} bytes): 로그인 HTML일 가능성'
    assert zipfile.is_zipfile(z), f'{z}는 정상 zip이 아님'
    tmp = Path('/content/flame_unpack') / zip_name.replace('.zip', '')
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True)
    with zipfile.ZipFile(z) as archive:
        archive.extractall(tmp)
    candidates = list(tmp.rglob(marker))
    assert candidates, f'{zip_name} 안에서 {marker}를 찾지 못함'
    src_root = candidates[0].parent
    shutil.rmtree(destination, ignore_errors=True)
    shutil.copytree(src_root, destination)
    for rel in required:
        p = destination / rel
        assert p.exists() and p.stat().st_size > 0, f'필수 FLAME asset 없음: {p}'
    print(zip_name, '->', destination, 'PASS')

install_flame(
    'FLAME2020.zip', 'generic_model.pkl', assets / 'FLAME2020',
    ['generic_model.pkl', 'landmark_embedding.npy', 'FLAME_masks/FLAME_masks.pkl'],
)
install_flame(
    'FLAME2023.zip', 'flame2023_no_jaw.pkl', assets / 'FLAME2023',
    ['flame2023_no_jaw.pkl'],
)

## 6. Pixel3DMM 경로 설정 및 private 입력 확인

사진을 Drive `MyDrive/hair_app/inputs/`에 넣는다. 정면/좌우 3·4/좌우 profile/hairline 노출을 포함해 5장 이상 권장한다.

In [ ]:
import os, pathlib
cfg = pathlib.Path.home() / '.config' / 'pixel3dmm'
cfg.mkdir(parents=True, exist_ok=True)
(cfg / '.env').write_text(
    'PIXEL3DMM_CODE_BASE="/content/pixel3dmm"\n'
    'PIXEL3DMM_PREPROCESSED_DATA="/content/p3dmm_preprocessed"\n'
    'PIXEL3DMM_TRACKING_OUTPUT="/content/p3dmm_tracking"\n'
)
INPUT_PATH = '/content/drive/MyDrive/hair_app/inputs'
VID_NAME = os.path.basename(INPUT_PATH.rstrip('/'))
os.makedirs(INPUT_PATH, exist_ok=True)
imgs = sorted(f for f in os.listdir(INPUT_PATH) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
print('input:', INPUT_PATH, '| VID_NAME:', VID_NAME, '| images:', len(imgs), imgs)
assert len(imgs) >= 2, '최소 2장 필요. 품질 비교에는 5장 이상 권장.'

## 7. 사진별 canonical crop 및 전처리 검증

공식 video용 `static_crop`은 서로 다른 사진의 bbox를 절대 픽셀 좌표에서 평균내므로 독립 사진에 사용하지 않는다. 각 사진에서 얼굴을 따로 검출하고 위치·크기·roll만 정규화한 512×512 crop을 한 번 만든다. yaw/pitch는 보존하며 원본↔crop 변환 행렬을 저장한다.

In [ ]:
import subprocess, urllib.request
from pathlib import Path

CROP_SCRIPT = Path('/content/canonical_face_crop.py')
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/Leejuseop/hair_app/main/experiments/milestone1_geometry_bakeoff/canonical_face_crop.py',
    CROP_SCRIPT,
)
root = Path('/content/p3dmm_preprocessed') / VID_NAME
subprocess.run([
    'conda', 'run', '-n', 'p3dmm', 'python', str(CROP_SCRIPT),
    '--input-dir', INPUT_PATH,
    '--output-root', str(root),
    '--output-size', '512',
    '--bbox-margin', '1.42',
    '--detection-threshold', '0.5',
    '--clean-output',
], check=True)
print('per-image canonical crop: PASS')

### 7.1 원본/crop 시각 gate

모든 crop에서 헤어라인·눈·코·입·턱·양쪽 귀가 필요한 범위만큼 보여야 한다. 하나라도 잘렸으면 다음 셀을 실행하지 않는다.

In [ ]:
import math
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

original_files = sorted(
    p for p in Path(INPUT_PATH).iterdir()
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}
)
cropped_files = sorted((root / 'cropped').glob('*.jpg'))
assert len(original_files) == len(cropped_files)
fig, axes = plt.subplots(len(cropped_files), 2, figsize=(12, 4 * len(cropped_files)), squeeze=False)
for index, (source_path, crop_path) in enumerate(zip(original_files, cropped_files)):
    source = ImageOps.exif_transpose(Image.open(source_path)).convert('RGB')
    crop = Image.open(crop_path).convert('RGB')
    axes[index, 0].imshow(source)
    axes[index, 0].set_title(f'원본: {source_path.name}\n{source.width}×{source.height}')
    axes[index, 1].imshow(crop)
    axes[index, 1].set_title(f'canonical crop: {crop_path.name}\n{crop.width}×{crop.height}')
    axes[index, 0].axis('off'); axes[index, 1].axis('off')
plt.tight_layout(); plt.show()
print('VISUAL GATE: 위 crop을 직접 확인한 뒤 다음 셀로 이동')

### 7.2 canonical crop에서 PIPNet landmark와 FaRL segmentation 실행

PIPNet은 crop을 다시 하지 않고 canonical 512×512 입력을 그대로 사용한다. FaRL 재검출 제거는 crop 수정 결과를 먼저 분리 검증한 다음 별도 A/B로 진행한다.

In [ ]:
import shutil, subprocess
from pathlib import Path

# canonical crop에서는 upstream의 과도한 0.99 landmark gate를 detector crop gate와 같은 0.75로 낮춘다.
pipnet_utils = Path('/content/pixel3dmm/src/pixel3dmm/preprocessing/pipnet_utils.py')
text = pipnet_utils.read_text()
if 'HAIR_APP_CANONICAL_LANDMARK_THRESHOLD' not in text:
    old = "if detections[i][1] < 0.99:"
    assert old in text
    text = text.replace(old, "if detections[i][1] < 0.75:  # HAIR_APP_CANONICAL_LANDMARK_THRESHOLD")
crop_info_old = "if not os.path.exists(f'{image_dir}/../crop_ymin_ymax_xmin_xmax.npy'):"
if 'HAIR_APP_DISABLE_CROP_METADATA_GUARD' not in text:
    assert crop_info_old in text
    text = text.replace(
        crop_info_old,
        "if (not disable_cropping) and not os.path.exists(f'{image_dir}/../crop_ymin_ymax_xmin_xmax.npy'):  # HAIR_APP_DISABLE_CROP_METADATA_GUARD",
    )
pipnet_utils.write_text(text)

for name in ('PIPnet_landmarks', 'PIPnet_annotated_images', 'pipnet', 'seg_og', 'seg_non_crop_annotations'):
    shutil.rmtree(root / name, ignore_errors=True)

helper = Path('/content/run_pipnet_on_canonical.py')
helper.write_text("""
import sys
sys.path.insert(0, '/content/pixel3dmm/scripts')
from run_cropping import run
run(
    'experiments/WFLW/pip_32_16_60_r18_l2_l1_10_1_nb10.py',
    sys.argv[1],
    start_frame=-1,
    static_crop=True,
    max_bbox=True,
    disable_cropping=True,
)
""")
with open('/content/p3dmm_preprocess.log', 'w') as log:
    subprocess.run([
        'conda', 'run', '-n', 'p3dmm', 'python', str(helper), str(root / 'rgb')
    ], stdout=log, stderr=subprocess.STDOUT, check=True)
    subprocess.run([
        'conda', 'run', '-n', 'p3dmm', 'python',
        '/content/pixel3dmm/scripts/run_facer_segmentation.py',
        '--video_name', VID_NAME,
    ], stdout=log, stderr=subprocess.STDOUT, check=True)
print(Path('/content/p3dmm_preprocess.log').read_text()[-8000:])

In [ ]:
from pathlib import Path
root = Path('/content/p3dmm_preprocessed') / VID_NAME
cropped = sorted((root / 'cropped').glob('*.jpg'))
crop_meta = sorted(p for p in (root / 'crop_meta').glob('*.json') if p.name != 'manifest.json')
landmarks = sorted((root / 'PIPnet_landmarks').glob('*.npy'))
seg = sorted((root / 'seg_og').glob('*.png'))
print('input/crop/meta/landmark/seg:', len(imgs), len(cropped), len(crop_meta), len(landmarks), len(seg))
assert len(cropped) == len(imgs), 'crop 개수가 입력 개수와 다름: preprocess log 확인'
assert len(crop_meta) == len(cropped), 'crop transform metadata 누락'
assert len(landmarks) == len(cropped), 'PIPNet landmark 누락: preprocess log 확인'
assert len(seg) == len(cropped), 'segmentation 개수가 crop 개수와 다름: preprocess log 확인'
assert all(p.stat().st_size > 0 for p in cropped + crop_meta + landmarks + seg)
print('preprocessing completeness: PASS')

## 8. Pixel3DMM network inference 및 출력 개수 검증

공식 script는 frame 내부 exception을 출력하고 계속 진행하므로 shell exit code만 믿지 않고 결과 개수를 검사한다.

In [ ]:
%%bash -s "$VID_NAME"
set -euo pipefail
cd /content/pixel3dmm
conda run -n p3dmm python scripts/network_inference.py model.prediction_type=normals video_name="$1" 2>&1 | tee /content/p3dmm_normals.log
conda run -n p3dmm python scripts/network_inference.py model.prediction_type=uv_map video_name="$1" 2>&1 | tee /content/p3dmm_uv.log

In [ ]:
from pathlib import Path
root = Path('/content/p3dmm_preprocessed') / VID_NAME
expected = len(list((root / 'cropped').glob('*')))
normal_files = list((root / 'p3dmm' / 'normals').glob('*.png'))
uv_files = list((root / 'p3dmm' / 'uv_map').glob('*.png'))
print('expected/normals/uv:', expected, len(normal_files), len(uv_files))
assert len(normal_files) == expected, 'normal inference 일부/전체 실패: /content/p3dmm_normals.log 확인'
assert len(uv_files) == expected, 'UV inference 일부/전체 실패: /content/p3dmm_uv.log 확인'
print('network inference completeness: PASS')

## 9. Multi-image FLAME tracking

공식 README의 `iters=100 iters=1500` 중복은 config 설명에 맞춰 `iters=100 global_iters=1500`으로 교정했다. 입력 수보다 큰 default batch 16 때문에 crash하지 않도록 batch를 동적으로 낮춘다.

In [ ]:
%%bash -s "$VID_NAME"
set -euo pipefail
FLAME=/content/pixel3dmm/src/pixel3dmm/preprocessing/MICA/data
test -s "$FLAME/FLAME2020/generic_model.pkl"
test -s "$FLAME/FLAME2020/landmark_embedding.npy"
test -s "$FLAME/FLAME2020/FLAME_masks/FLAME_masks.pkl"
test -s "$FLAME/FLAME2023/flame2023_no_jaw.pkl"
N=$(find "/content/p3dmm_preprocessed/$1/cropped" -maxdepth 1 -type f | wc -l)
test "$N" -ge 2
BATCH=$N
if [ "$BATCH" -gt 16 ]; then BATCH=16; fi
echo "views=$N batch_size=$BATCH"
cd /content/pixel3dmm
conda run -n p3dmm python scripts/track.py video_name="$1" \
  iters=100 global_iters=1500 batch_size=$BATCH \
  include_neck=False w_exp=0.1 use_mouth_lmk=False \
  w_shape=0.01 w_shape_general=0.001 normal_super=2000.0 sil_super=1000.0 \
  use_flame2023=True ignore_mica=True is_discontinuous=True \
  2>&1 | tee /content/p3dmm_tracking.log

## 10. 3D mesh 확인

In [ ]:
!pip -q install trimesh plotly
import glob, trimesh
import plotly.graph_objects as go
cands = []
for ext in ('ply', 'obj'):
    cands += glob.glob(f'/content/p3dmm_tracking/**/*.{ext}', recursive=True)
cands = sorted(cands)
print('meshes:', len(cands), *cands[-10:], sep='\n')
assert cands, 'mesh 없음: /content/p3dmm_tracking.log의 첫 traceback 확인'
mesh_path = cands[-1]
mesh = trimesh.load(mesh_path, force='mesh')
assert len(mesh.vertices) and len(mesh.faces), 'mesh가 비어있음'
v, f = mesh.vertices, mesh.faces
fig = go.Figure(data=[go.Mesh3d(x=v[:,0], y=v[:,1], z=v[:,2], i=f[:,0], j=f[:,1], k=f[:,2], color='lightgray', flatshading=True)])
fig.update_layout(scene=dict(aspectmode='data'), margin=dict(l=0,r=0,t=0,b=0))
fig.show()
print('mesh preview PASS:', mesh_path, '| vertices:', len(v), '| faces:', len(f))

## 11. 결과, raw logs, manifest를 Drive에 저장

성공·실패 여부와 관계없이 이 셀을 실행하면 다음 디버깅에 필요한 log를 보존한다.

In [ ]:
import datetime, glob, json, pathlib, shutil, subprocess, torch
base = pathlib.Path('/content/drive/MyDrive/hair_app')
run_id = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_dir = base / 'runs' / f'pixel3dmm_{VID_NAME}_{run_id}'
(run_dir / 'logs').mkdir(parents=True, exist_ok=True)
(run_dir / 'meshes').mkdir(parents=True, exist_ok=True)
for log in glob.glob('/content/p3dmm_*.log'):
    shutil.copy2(log, run_dir / 'logs' / pathlib.Path(log).name)
mesh_files = []
for ext in ('ply', 'obj'):
    mesh_files += glob.glob(f'/content/p3dmm_tracking/**/*.{ext}', recursive=True)
for mesh_file in mesh_files:
    shutil.copy2(mesh_file, run_dir / 'meshes' / pathlib.Path(mesh_file).name)
commit = subprocess.run(['git','-C','/content/pixel3dmm','rev-parse','HEAD'], capture_output=True, text=True).stdout.strip()
manifest = {
    'model': 'pixel3dmm',
    'commit': commit,
    'license': 'CC BY-NC 4.0 (non-commercial)',
    'gpu': torch.cuda.get_device_name(0),
    'compute_capability': list(torch.cuda.get_device_capability(0)),
    'torch_cuda_arch_list': pathlib.Path('/content/p3dmm_torch_arch.txt').read_text().strip(),
    'input_set_id': VID_NAME,
    'input_count': len(imgs),
    'tracking_config': 'iters=100 global_iters=1500 dynamic_batch include_neck=False w_exp=0.1 use_mouth_lmk=False w_shape=0.01 w_shape_general=0.001 normal_super=2000 sil_super=1000 use_flame2023=True ignore_mica=True is_discontinuous=True',
    'fixes_applied': [
        'https clones', 'Cython before FaceBoxes', 'correct checkpoint paths',
        'facer image_ids.long', 'skip MICA preprocessing', 'zero MICA shape prior',
        'manual validated FLAME2020+FLAME2023', 'dynamic batch size',
        'corrected duplicate iters to global_iters', 'output count validation',
    ],
    'mesh_count': len(mesh_files),
    'created_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
(run_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print('saved:', run_dir)
print(json.dumps(manifest, indent=2, ensure_ascii=False))

## 12. 평가

`scoring_sheet.csv`에 identity, geometry, hairline, side contour, scalp/ear topology, execution reliability를 1–5로 기록한다. Hidden scalp/rear는 측정값이 아니라 prior 추정이다.